# Un script permettant de calculer des carte de likelihood pour des réseaux pré-entrainés sur Imagenet et testés sur le dataset Animal 10k


In [1]:
from retinotopy import *

Welcome on Linux-6.5.0-1019-oem-x86_64-with-glibc2.35
CONEC-LID-001


Running on GPU :  NVIDIA RTX A2000 12GB #GPU= 1


[nltk_data] Downloading package wordnet to
[nltk_data]     /home/INT/perrinet.l/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


On date 2024-04-19, Running learning on host CONEC-LID-001 with device cuda


In [2]:

@dataclass
class Params:
    DEBUG = 1
    
    datetag: str = datetag # Set the date of the result's file
    loader: str = f'{DATAROOT}/Imagenet_urls_ILSVRC_2016.json' # File containing Imagenet's labels
    annotations: str = f'{DATAROOT}/animal_10k/ap-10k/annotations/clean_annotations.json' # File containing Imagenet's labels
    model_path: str = f'{data_cache}' # Set the parth to store the re-trained model
    root: str = f'{DATAROOT}/animal_10k/ap-10k/animal_data' # Directory containing images to perform the training
    #root: str = f'{DATAROOT}/animal_10k/ap-10k/' # Directory containing images to perform the training
    #annotations: str = f'{DATAROOT}/animal_10k/annotations/clean_annotations.json' # File containing Imagenet's labels
    
    
    folders: list = field(default_factory=lambda: ['val']) # Set the training and validation folders relative to the root
    seed: int = 667 # Set the seed for reproducibility 
    #seed: int = np.random.choice(1000) # random
    rs_min: float = 0.05
    rs_max: float = -4.95
    do_polar: bool = True
    do_rottrain: bool = True
    
    torch.manual_seed(seed)
    image_size: int = 224 # Set the training and validation folders relative to the root

print()

args = Params()
datetag = args.datetag

In [3]:
image_size = 224
image_size_retino = [image_size, image_size]
image_size_lin = [image_size, image_size]
resolution = (10,10)

def get_start(start_value):
    return torch.log2(torch.tensor(start_value)).item()

center = -4.95
sample_sizes = [1/7, 0.2, 0.3, 0.4, 0.5]

def get_retino_grid(center=center, image_size_retino=image_size_retino, resolution=resolution, sample_sizes=sample_sizes, base_point=False):
    grids = []
    for sample_size in sample_sizes:
        start = get_start(sample_size)
        rs_ = torch.logspace(start, center, image_size_retino[0], base = 2)
        ts_ = torch.linspace(0, torch.pi*2, image_size_retino[1])
    
        grid_xs = torch.outer(rs_, torch.cos(ts_)) 
        grid_ys = torch.outer(rs_, torch.sin(ts_)) 
        for i in np.linspace(-(1-sample_size), (1-sample_size), (resolution[0])):
            for j in np.linspace(-(1-sample_size), (1-sample_size), (resolution[1])):
                grids.append(torch.stack((grid_xs+j, grid_ys+i), 2))

    if base_point : 
        rs_ = torch.logspace(0, -5, image_size_retino[0], base = 2)
        grid_xs = torch.outer(rs_, torch.cos(ts_)) 
        grid_ys = torch.outer(rs_, torch.sin(ts_)) 
        grids.append(torch.stack((grid_xs, grid_ys), 2))
    
    return torch.stack(grids)

retino_grid = get_retino_grid(center=center, image_size_retino=image_size_retino, resolution=resolution, sample_sizes=sample_sizes)

def get_lin_grid(sample_sizes=sample_sizes, image_size_lin=image_size_lin, resolution=resolution):
    grids = []
    for sample_size in sample_sizes:
        x = torch.linspace(-sample_size, sample_size, image_size_lin[0])
        y = torch.linspace(-sample_size, sample_size, image_size_lin[1])
        grid_x, grid_y = torch.meshgrid(x, y, indexing='ij')
        
        for i in np.linspace(-(1-sample_size), (1-sample_size), (resolution[0])):
            for j in np.linspace(-(1-sample_size), (1-sample_size), (resolution[1])):
                grids.append(torch.stack((grid_y+j, grid_x+i), 2))
        
    return torch.stack(grids)

lin_grid = get_lin_grid(sample_sizes=sample_sizes, image_size_lin=image_size_lin, resolution=resolution)

def to_retino_tens_grid(images, retino_grid): 
    return nnf.grid_sample(images.float(), retino_grid, align_corners=False)#, padding_mode='border')

class to_retino_tens(object): 
    def __init__(self, retino_grid):
        self.grid = retino_grid
        

    def __call__(self, images):
        images = images.repeat(retino_grid.shape[0],1,1,1)
        return nnf.grid_sample(images, self.grid, 
                               padding_mode="border", align_corners=False).squeeze(dim=0)

In [4]:
test_path = "../2023-12-06_Retinotopy/"

    
match = []
labels = []

#----------------Get the label for the Imagenet categorization------------------------
for i_img, img_id in enumerate(Imagenet_urls_ILSVRC_2016):
    syn_= wn.synset_from_pos_and_offset('n', int(img_id.replace('n','')))
    sem_ = syn_.hypernym_paths()[0]
    labels.append(syn_)
    for i in np.arange(len(sem_)):
        if sem_[i].lemmas()[0].name() in 'animal' :
            match.append(i_img)

#------------------------------------------------------------------------------------  


#2D Gaussian function
def twoD_Gaussian(x, y, xo, yo, sigma_x, sigma_y):
    a = 1./(2*sigma_x**2) + 1./(2*sigma_y**2)
    c = 1./(2*sigma_x**2) + 1./(2*sigma_y**2)
    g = np.exp( - (a*((x-xo)**2) + c*((y-yo)**2)))
    return g.ravel()


def datasets_transforms(args, grid, im_mean=im_mean, im_std=im_std,
                        num_workers=num_workers, pin_memory=True, verbose=True):


    dataloaders = {}
    transforms = [                
        T.ToImage(),  # Convert to tensor, only needed if you had a PIL image

        T.ToDtype(torch.float32, scale=True),  # Normalize expects float input
    ]   
 
    transforms.append(to_retino_tens(grid))
#    transforms.append(T.Resize((int(224), int(224)), interpolation=interpolation, antialias=True))

    transforms.append(T.Normalize(mean=im_mean, std=im_std)) # to normalize colors on the imagenet dataset
    data_transform = T.Compose(transforms)

    image_dataset = datasets.ImageFolder(args.root, transform=data_transform) # load the data

    dataloaders = torch.utils.data.DataLoader(
                            image_dataset, 
                            batch_size= 1, shuffle=True, 
                            num_workers=num_workers, pin_memory=pin_memory
                    )
    if True: 
        print(f"Loaded {len(image_dataset)} images")  

    return dataloaders, image_dataset


In [5]:
model_paths = {}
models = {}

for model in ['resnet101']:
    for retino in [True, False]:
        model_name = model+'_'+str(retino)
        model_paths[model_name] = f"{test_path}cached_data/2024-03-04_{model}_do_polar={retino}_do_rottrain=False.pt"
        models[model_name] = charge_model(model_paths[model_name]).eval()

loading .... ../2023-12-06_Retinotopy/cached_data/2024-03-04_resnet101_do_polar=True_do_rottrain=False.pt


FileNotFoundError: [Errno 2] No such file or directory: '../2023-12-06_Retinotopy/cached_data/2024-03-04_resnet101_do_polar=True_do_rottrain=False.pt'

In [6]:
for model_name in models.keys():

    grid = retino_grid if 'True' in model_name else lin_grid
    grid = lin_grid
    dataloader, image_dataset = datasets_transforms(args, grid)
    print(model_name)
    df_means = []
    for samples in np.linspace(0,4,5, dtype=int):
        df_means.append(pd.DataFrame([], columns=['model', 'image_name', 'likelihood_in_max', 'likelihood_out_max', 'dist',
                                            'likelihood_in_mean', 'likelihood_out_mean', 'mid_point', 'in_point', 'ext_point',
                                            'Iou', 'sum', 'sum_in', 'time', 'label']) )
        
    for i_image, (data, label) in enumerate(image_dataset):
    
        image_name = image_dataset.imgs[i_image][0].split('/')[-1]
        boxes = annotations[image_name]['keypoints']
        origin_size = [annotations[image_name]['image_info']['height'], annotations[image_name]['image_info']['width']]
        brut_array = to_heatmap(boxes, origin_size)
        normalized_array = normalize_array(brut_array)
        ground_true = enlarge_function(normalized_array.reshape(origin_size), resolution) > 0.2

    
        if len(np.where(ground_true > 0)[0]) == 0 :
            try:
                for boxe in boxes:
                    coord = (little_box(boxe, origin_size, resolution))
                    ground_true[coord[1],coord[0]] = 1
            except:
                continue
                
        ground_true_indices = np.where(ground_true.reshape(resolution[0]*resolution[1]) > 0)

        three_points = get_three_points(ground_true_indices, resolution)


        if len(ground_true_indices[0]) != resolution[0]*resolution[1]:
        
            with torch.no_grad():
                
                data, models[model_name] = data.to(device), models[model_name].to(device)
                
                since_heat = time.time()
                
                heatmap = models[model_name](data).squeeze(1)

                elapsed_time = time.time() - since_heat

                for samples in np.linspace(0,4,5, dtype=int):

                    heatmap_temp = heatmap[samples*(resolution[0]*resolution[1]) : (samples+1)*(resolution[0]*resolution[1])]
                    
                    sum = 1 if torch.argmax(torch.sum(heatmap_temp, dim=0)).item() in match else 0
                    sum_in = 1 if torch.argmax(torch.sum(heatmap_temp[ground_true_indices[0]], dim=0)).item() in match else 0
                    
                    heatmap_temp = torch.sum(torch.nn.functional.softmax(heatmap_temp, dim=1)[:,match], dim=1)
                    heatmap_temp = np.array(heatmap_temp.cpu().detach().numpy())

                    mid_point, in_point, ext_point = get_like_point(heatmap_temp, resolution, three_points)
                    
                    
                    out_heat = np.delete(heatmap_temp, [ground_true_indices[0]])
                    
                    likelihood_out_max = np.max(out_heat)
                    likelihood_in_max = np.max(heatmap_temp[ground_true_indices[0]])
                
                    likelihood_out_mean = np.mean(out_heat)
                    likelihood_in_mean = np.mean(heatmap_temp[ground_true_indices[0]])
                    
                
    
                    Iou = get_IoU(heatmap_temp, ground_true.reshape(resolution[0]*resolution[1]))
                    dist = [euclidean_distance(three_points[0], three_points[1]), euclidean_distance(three_points[1], three_points[2])]
                    dist[1] += dist[0] 
                    
                    print("\r{} /100 ".format(round((i_image/len(image_dataset))*100)), end="")
                    
                    df_means[samples].loc[len(df_means[samples])] = {'model':model_name, 'image_name':image_name, 'likelihood_in_max': likelihood_in_max, 'likelihood_out_max': likelihood_out_max,
                                                 'likelihood_out_mean': likelihood_out_mean, 'likelihood_in_mean': likelihood_in_mean, 'dist':dist,
                                                 'mid_point':mid_point, 'in_point':in_point, 'ext_point':ext_point, 'Iou': Iou,
                                                 'sum':sum,'sum_in':sum_in, 'time':elapsed_time, 'label':label}
    for samples in np.linspace(0,4,5, dtype=int):
        df_means[samples].to_csv(f'{data_cache}/{datetag}_likelihood_map_{model_name}_sample:{samples}.csv')
    models[model_name].cpu()